# 🛒 E-commerce Sales Analysis

**Objective:** Analyze transaction data to identify revenue trends, category/product performance, regional performance, channel contribution and operational issues.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

## 2. Load Data

In [ ]:
df = pd.read_csv('ecommerce_sales.csv')
print('Dataset shape:', df.shape)
df.head()

## 3. Data Quality Assessment

In [ ]:
quality = pd.DataFrame({
    'missing_values': df.isna().sum(),
    'data_type': df.dtypes.astype(str)
})
print('Duplicate rows:', df.duplicated().sum())
quality

## 4. Data Cleaning

In [ ]:
df = df.drop_duplicates().copy()
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')
df['region'] = df['region'].fillna('Unknown')
df['payment_method'] = df['payment_method'].fillna('Unknown')
df['discount_pct'] = df['discount_pct'].fillna(df['discount_pct'].median())
df['revenue'] = pd.to_numeric(df['revenue'], errors='coerce').fillna(0)
df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce').fillna(0).astype(int)
df['month'] = df['order_date'].dt.to_period('M').astype(str)

df.to_csv('cleaned_sales.csv', index=False)
print('Cleaned rows:', len(df))
df.head()

## 5. KPI Dashboard

In [ ]:
completed = df[df['order_status'] == 'Completed'].copy()

total_revenue = completed['revenue'].sum()
total_orders = completed['order_id'].nunique()
unique_customers = completed['customer_id'].nunique()
units_sold = completed['quantity'].sum()
aov = total_revenue / total_orders
cancel_rate = (df['order_status'] == 'Cancelled').mean() * 100
return_rate = (df['order_status'] == 'Returned').mean() * 100

kpis = pd.DataFrame({
    'KPI': ['Revenue', 'Completed Orders', 'Unique Customers', 'Units Sold', 'AOV', 'Cancellation Rate', 'Return Rate'],
    'Value': [f'₹{total_revenue:,.2f}', f'{total_orders:,}', f'{unique_customers:,}',
              f'{units_sold:,}', f'₹{aov:,.2f}', f'{cancel_rate:.2f}%', f'{return_rate:.2f}%']
})
kpis

## 6. Monthly Revenue & MoM Growth

In [ ]:
monthly = completed.groupby('month', as_index=False).agg(
    revenue=('revenue','sum'),
    orders=('order_id','nunique'),
    units=('quantity','sum')
)
monthly['mom_growth_pct'] = monthly['revenue'].pct_change() * 100
monthly

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(monthly['month'], monthly['revenue'], marker='o')
plt.title('Monthly Revenue Trend')
plt.xlabel('Month')
plt.ylabel('Revenue (INR)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 7. Category Analysis

In [ ]:
category = completed.groupby('category', as_index=False).agg(
    revenue=('revenue','sum'),
    orders=('order_id','nunique'),
    units=('quantity','sum')
)
category['revenue_share_pct'] = category['revenue'] / category['revenue'].sum() * 100
category.sort_values('revenue', ascending=False)

## 8. Regional Analysis

In [ ]:
region = completed.groupby('region', as_index=False).agg(
    revenue=('revenue','sum'),
    orders=('order_id','nunique')
).sort_values('revenue', ascending=False)
region

## 9. Sales Channel Analysis

In [ ]:
channel = completed.groupby('channel', as_index=False).agg(
    revenue=('revenue','sum'),
    orders=('order_id','nunique'),
    units=('quantity','sum')
)
channel['aov'] = channel['revenue'] / channel['orders']
channel.sort_values('revenue', ascending=False)

## 10. Top Products

In [ ]:
top_products = completed.groupby('product', as_index=False).agg(
    revenue=('revenue','sum'),
    units=('quantity','sum'),
    orders=('order_id','nunique')
).sort_values('revenue', ascending=False).head(10)
top_products

## 11. Customer Analysis

In [ ]:
customer = completed.groupby('customer_id', as_index=False).agg(
    revenue=('revenue','sum'),
    orders=('order_id','nunique'),
    units=('quantity','sum')
).sort_values('revenue', ascending=False)
customer.head(10)

## 12. Order Status Analysis

In [ ]:
status = df['order_status'].value_counts().rename_axis('status').reset_index(name='orders')
status['share_pct'] = status['orders'] / len(df) * 100
status

## 13. Key Business Questions

- Which month has the highest revenue?
- Which category contributes the most revenue?
- Which products generate the most revenue?
- Which region and channel perform best?
- What is the average order value?
- What are the cancellation and return rates?
- Which customers contribute the most revenue?

## Conclusion

The project demonstrates an end-to-end Python analytics workflow from data quality assessment and cleaning to KPI calculation, segmentation, visualization and business recommendations.